<a href="https://colab.research.google.com/github/25ds3000042/GPU_Programming_Fundamentals_with_CUDA/blob/main/GPU_enabled_Sample_CUDA_Python_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Numba
!pip install -q numba

from numba import cuda
import numpy as np
import time

In [2]:
# Check GPU
print("CUDA available:", cuda.is_available())

if cuda.is_available():
    print("GPU:", cuda.get_current_device().name)
else:
    print("GPU not available. Enable T4 GPU in Colab.")


CUDA available: True
GPU: Tesla T4


In [3]:
# CUDA kernel
@cuda.jit
def vector_add_gpu(a, b, c):
    i = cuda.grid(1)

    if i < a.size:
        c[i] = a[i] + b[i]


In [4]:
# Create data
N = 1_000_000

a = np.random.rand(N).astype(np.float32)
b = np.random.rand(N).astype(np.float32)
c = np.zeros(N, dtype=np.float32)


if cuda.is_available():

    # Copy data CPU → GPU
    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_c = cuda.device_array(N, dtype=np.float32)

    # Threads and blocks
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Start GPU timer
    start = time.time()

    # Launch CUDA kernel
    vector_add_gpu[blocks_per_grid, threads_per_block](
        d_a, d_b, d_c
    )

    # Wait for GPU to finish
    cuda.synchronize()

    # Copy GPU → CPU
    c = d_c.copy_to_host()

    end = time.time()

    print("GPU computation time:", end - start, "seconds")

    # Verify result
    print("First 10 results:")
    print(c[:10])

    print("\nCorrect:",
          np.allclose(c, a + b))

GPU computation time: 1.6368329524993896 seconds
First 10 results:
[0.87830365 0.8775847  0.3555264  0.9263144  0.9657003  0.16105387
 0.6614146  0.5485067  1.2007084  0.63167024]

Correct: True
